In [1]:
import json, time, gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

RAW_ROOT  = Path("data/raw")
PROC_DIR  = Path("data/processed")
FUZZY_DIR = Path("data/fuzzy")
FUZZY_DIR.mkdir(parents=True, exist_ok=True)

with open(PROC_DIR / "task_registry.json") as f:
    TASKS = json.load(f)
with open(PROC_DIR / "feature_list.json") as f:
    FEAT = json.load(f)

FEATURES    = FEAT["features"]
CATEGORICAL = [c for c in ["userCoinTypeMode", "userReserveMode"] if c in FEATURES]

HOLDOUT_START = pd.Timestamp("2024-04-01").timestamp()

# IT2FIS parameters
N_REGIMES  = 4
REGIME_NAMES = ["Safe", "Watch", "Danger", "Critical"]
FOU_WIDTH  = 0.20
FCM_M      = 2.0
SEED       = 42
np.random.seed(SEED)

In [2]:
def build_orthogonal_signals(df):
    """
    Build 4 signals that are mathematically derived from the 90 base features but
    are not themselves in that set — ratios, user-vs-market, extremeness of amount.
    """
    s = pd.DataFrame(index=df.index)

    # (1) market stress: ratio of market liquidation activity to total market activity
    tot_mkt = (df["marketBorrowCount"].fillna(0) + df["marketDepositCount"].fillna(0) +
               df["marketWithdrawCount"].fillna(0) + df["marketRepayCount"].fillna(0) + 1e-9)
    s["mkt_liq_share"] = df["marketLiquidationCount"].fillna(0) / tot_mkt

    # (2) user vs market concentration: how much of market activity is from this user
    tot_user = (df["userBorrowCount"].fillna(0) + df["userDepositCount"].fillna(0) +
                df["userWithdrawCount"].fillna(0) + df["userRepayCount"].fillna(0) + 1e-9)
    s["user_mkt_ratio"] = tot_user / (tot_mkt + 1)

    # (3) amount extremeness: current amount relative to user's own average
    user_hist = (df["userBorrowAvgAmountUSD"].fillna(0) + df["userDepositAvgAmountUSD"].fillna(0) +
                 df["userRepayAvgAmountUSD"].fillna(0) + df["userWithdrawAvgAmountUSD"].fillna(0)) / 4 + 1e-9
    s["amount_z"] = np.log1p(df["amountUSD"].fillna(0)) - np.log1p(user_hist)

    # (4) user liquidation pressure: fraction of user's lifetime that was liquidated
    s["user_liq_rate"] = df["userLiquidationCount"].fillna(0) / (tot_user + 1)

    return s

# Smoke test on first task
task = TASKS[0]
sample = pd.read_csv(RAW_ROOT / task["train_path"], nrows=10_000,
                     usecols=FEATURES + CATEGORICAL + ["timeDiff","status","timestamp"])
sig = build_orthogonal_signals(sample)
print(sig.describe().round(4))
print(f"\nnon-finite counts: {(~np.isfinite(sig)).sum().to_dict()}")

       mkt_liq_share  user_mkt_ratio    amount_z  user_liq_rate
count     10000.0000      10000.0000  10000.0000     10000.0000
mean          0.0015          0.0556      0.0482         0.0002
std           0.0008          0.0779      0.7752         0.0041
min           0.0000          0.0000     -7.9417         0.0000
25%           0.0011          0.0002     -0.1166         0.0000
50%           0.0017          0.0016      0.2188         0.0000
75%           0.0021          0.1177      0.4364         0.0000
max           0.0023          0.2361      3.0839         0.2000

non-finite counts: {'mkt_liq_share': 0, 'user_mkt_ratio': 0, 'amount_z': 0, 'user_liq_rate': 0}


In [3]:
def fuzzy_cmeans(X, K, m=2.0, max_iter=100, tol=1e-5, seed=42):
    rng = np.random.default_rng(seed)
    N, D = X.shape
    U = rng.dirichlet(np.ones(K), size=N)
    for _ in range(max_iter):
        Um = U ** m
        C = (Um.T @ X) / Um.sum(axis=0)[:, None]
        dist = np.linalg.norm(X[:, None, :] - C[None, :, :], axis=2) + 1e-12
        inv_m = (1.0 / dist) ** (2.0 / (m - 1))
        U_new = inv_m / inv_m.sum(axis=1, keepdims=True)
        if np.linalg.norm(U_new - U) < tol:
            U = U_new; break
        U = U_new
    return C, U

def it2_memberships(X, centres, sigmas_per_centre, fou=FOU_WIDTH):
    N, K = len(X), len(centres)
    mu_upper = np.zeros((N, K))
    mu_lower = np.zeros((N, K))
    for k in range(K):
        d2 = np.sum((X - centres[k])**2, axis=1)
        sig = sigmas_per_centre[k]
        mu_upper[:, k] = np.exp(-d2 / (2 * (sig * (1 + fou))**2))
        mu_lower[:, k] = np.exp(-d2 / (2 * (sig * (1 - fou))**2))
    mu_mid = (mu_upper + mu_lower) / 2
    mu_hat = mu_mid / (mu_mid.sum(axis=1, keepdims=True) + 1e-12)
    interval_width = (mu_upper - mu_lower).sum(axis=1)
    return mu_hat, interval_width

In [4]:
# Pick first task for fitting (centres transfer across tasks since inputs are feature-space)
task = TASKS[0]
print(f"fitting FCM on orthogonal signals from task {task['task_id']} (fit window only)")

cols = FEATURES + CATEGORICAL + ["timestamp"]
cols = list(dict.fromkeys(cols))
df_all = pd.read_csv(RAW_ROOT / task["train_path"], usecols=cols)
df_all = df_all[df_all["timestamp"] < HOLDOUT_START].reset_index(drop=True)
print(f"fit-window rows: {len(df_all):,}")

sig = build_orthogonal_signals(df_all)
sig = sig.replace([np.inf, -np.inf], np.nan).dropna()
# Downsample for FCM speed (500k rows is enough to estimate 4 cluster centres)
sample_idx = np.random.default_rng(SEED).choice(len(sig), size=min(500_000, len(sig)), replace=False)
X_fit = sig.iloc[sample_idx].values

# Standardize using fit-sample stats only
mu_z = X_fit.mean(axis=0)
sd_z = X_fit.std(axis=0) + 1e-8
X_fit_sc = (X_fit - mu_z) / sd_z

print("running fuzzy c-means (4 regimes)...")
centres, U = fuzzy_cmeans(X_fit_sc, K=N_REGIMES, m=FCM_M, seed=SEED)

# Per-centre spread for IT2 FOU
sigmas = np.array([np.std(np.linalg.norm(X_fit_sc - c, axis=1)) + 1e-8 for c in centres])
print(f"regime centres (scaled):\n{centres.round(3)}")
print(f"per-regime sigmas: {sigmas.round(3)}")

# Order regimes by "stress level": heuristic = centre's position on mkt_liq_share axis (col 0)
order = np.argsort(centres[:, 0])
centres = centres[order]
sigmas = sigmas[order]
print(f"regimes ordered Safe→Critical by mkt_liq_share")

# Save FCM artifacts
np.save(FUZZY_DIR / "centres.npy", centres)
np.save(FUZZY_DIR / "sigmas.npy", sigmas)
np.save(FUZZY_DIR / "signal_scaler.npy", np.stack([mu_z, sd_z]))
print(f"saved: centres, sigmas, scaler → {FUZZY_DIR}")

fitting FCM on orthogonal signals from task Borrow_Deposit (fit window only)
fit-window rows: 784,167
running fuzzy c-means (4 regimes)...
regime centres (scaled):
[[-0.221 -0.315 -1.096  0.006]
 [-0.146  2.809  0.61  -0.034]
 [ 2.481 -0.103 -0.008  0.023]
 [-0.327 -0.329  0.331 -0.049]]
per-regime sigmas: [1.432 1.275 1.213 1.6  ]
regimes ordered Safe→Critical by mkt_liq_share
saved: centres, sigmas, scaler → data\fuzzy


In [5]:
def compute_fuzzy_cols_for_task(task):
    cols = FEATURES + CATEGORICAL + ["timeDiff","status","timestamp"]
    cols = list(dict.fromkeys(cols))
    df = pd.read_csv(RAW_ROOT / task["train_path"], usecols=cols)
    df[FEATURES] = df[FEATURES].fillna(0)

    sig = build_orthogonal_signals(df).replace([np.inf,-np.inf], np.nan).fillna(0)
    X_sc = (sig.values - mu_z) / sd_z
    mu_hat, iw = it2_memberships(X_sc, centres, sigmas, fou=FOU_WIDTH)

    fuzzy_cols = pd.DataFrame(mu_hat, columns=[f"mu_{n}" for n in REGIME_NAMES])
    fuzzy_cols["interval_width"] = iw
    fuzzy_cols["timestamp"] = df["timestamp"].values
    return fuzzy_cols

t_start = time.time()
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    print(f"[{i:2d}/16] {tid}  computing memberships", end=" ")
    t0 = time.time()
    fcols = compute_fuzzy_cols_for_task(task)
    fcols.to_parquet(FUZZY_DIR / f"{tid}_memberships.parquet")
    print(f"({len(fcols):,} rows, {time.time()-t0:.0f}s)")
    del fcols; gc.collect()

print(f"\nall memberships saved in {(time.time()-t_start)/60:.1f} min")

[ 1/16] Borrow_Deposit  computing memberships (882,736 rows, 10s)
[ 2/16] Borrow_Repay  computing memberships (846,310 rows, 9s)
[ 3/16] Borrow_Withdraw  computing memberships (883,952 rows, 9s)
[ 4/16] Borrow_Liquidated  computing memberships (885,908 rows, 9s)
[ 5/16] Deposit_Borrow  computing memberships (2,168,050 rows, 24s)
[ 6/16] Deposit_Repay  computing memberships (2,165,549 rows, 21s)
[ 7/16] Deposit_Withdraw  computing memberships (1,970,714 rows, 19s)
[ 8/16] Deposit_Liquidated  computing memberships (2,171,234 rows, 21s)
[ 9/16] Repay_Borrow  computing memberships (740,589 rows, 7s)
[10/16] Repay_Deposit  computing memberships (776,197 rows, 8s)
[11/16] Repay_Withdraw  computing memberships (780,761 rows, 7s)
[12/16] Repay_Liquidated  computing memberships (781,837 rows, 8s)
[13/16] Withdraw_Borrow  computing memberships (1,586,156 rows, 16s)
[14/16] Withdraw_Deposit  computing memberships (1,388,700 rows, 15s)
[15/16] Withdraw_Repay  computing memberships (1,587,042 rows,

In [6]:
# Load one task's memberships and validate
sample = pd.read_parquet(FUZZY_DIR / f"{TASKS[0]['task_id']}_memberships.parquet")
in_fit = sample["timestamp"] < HOLDOUT_START

# Regime distribution in fit window
hard_regime = sample[[f"mu_{n}" for n in REGIME_NAMES]].values.argmax(axis=1)
print("regime distribution in fit window:")
for k, name in enumerate(REGIME_NAMES):
    frac = (hard_regime[in_fit] == k).mean() * 100
    print(f"  {name:10s}  {frac:5.2f}%")

print("\nregime distribution in holdout window:")
for k, name in enumerate(REGIME_NAMES):
    frac = (hard_regime[~in_fit] == k).mean() * 100
    print(f"  {name:10s}  {frac:5.2f}%")

print(f"\nmean interval width (fit):     {sample.loc[in_fit, 'interval_width'].mean():.4f}")
print(f"mean interval width (holdout): {sample.loc[~in_fit, 'interval_width'].mean():.4f}")

regime distribution in fit window:
  Safe        66.66%
  Watch       18.58%
  Danger       6.77%
  Critical     7.99%

regime distribution in holdout window:
  Safe        40.89%
  Watch       37.34%
  Danger       0.00%
  Critical    21.77%

mean interval width (fit):     0.5242
mean interval width (holdout): 0.5887


In [7]:
# Show the actual regime centres and distances between them
centres = np.load("data/fuzzy/centres.npy")
sigmas  = np.load("data/fuzzy/sigmas.npy")
print("centre coordinates (scaled space):")
print(pd.DataFrame(centres, columns=["mkt_liq_share","user_mkt_ratio","amount_z","user_liq_rate"],
                   index=["Safe","Watch","Danger","Critical"]).round(3))
print(f"\nsigmas: {sigmas.round(3)}")

# Pairwise distances between centres
from scipy.spatial.distance import cdist
print("\npairwise centre distances:")
print(pd.DataFrame(cdist(centres, centres).round(3),
                   index=["Safe","Watch","Danger","Critical"],
                   columns=["Safe","Watch","Danger","Critical"]))

centre coordinates (scaled space):
          mkt_liq_share  user_mkt_ratio  amount_z  user_liq_rate
Safe             -0.327          -0.329     0.331         -0.049
Watch            -0.221          -0.315    -1.096          0.006
Danger           -0.146           2.809     0.610         -0.034
Critical          2.481          -0.103    -0.008          0.023

sigmas: [1.6   1.432 1.275 1.213]

pairwise centre distances:
           Safe  Watch  Danger  Critical
Safe      0.000  1.432   3.155     2.838
Watch     1.432  0.000   3.560     2.920
Danger    3.155  3.560   0.000     3.970
Critical  2.838  2.920   3.970     0.000
